# **Подготовка**

In [1]:
!pip install -r requirements.txt -q
!pip install tqdm -q

In [2]:
import os
import re
import math
import time
import pickle
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = Path("data")
DOCS_PATH = DATA_DIR / "vkmarco-docs.tsv"
TRAIN_Q_PATH = DATA_DIR / "vkmarco-doctrain-queries.tsv"
TRAIN_QRELS_PATH = DATA_DIR / "vkmarco-doctrain-qrels.tsv"
EVAL_Q_PATH = DATA_DIR / "vkmarco-doceval-queries.tsv"
OBJECTS_PATH = DATA_DIR / "objects.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

SEED = 42
np.random.seed(SEED)

/home/snomty/hse-2027-retrieval-and-ranking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_docs(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                parts = parts + [""] * (4 - len(parts))
            doc_id, url, title, body = parts[0], parts[1], parts[2], parts[3]
            rows.append((doc_id, url, title, body))
    return pd.DataFrame(rows, columns=["DocumentId", "URL", "Title", "Body"])

docs = load_docs(DOCS_PATH)
print("docs shape:", docs.shape)
docs.head(3)

docs shape: (74848, 4)


,DocumentId,URL,Title,Body
0,D000000001,zvuk.top/tracks/%D0%B4%D0%BE%D0%BC-raim-feat-a...,Дом raim feat artur adil музыка в MP3 🎵 – скач...,Скачивай и слушай 🎧 raim дом o2 2018 на Zvuk.t...
1,D000000002,musickz.kz/index.php?newsid=3221,Скачать RaiM feat. Artur & Adil - Дом 2018,Скачать песню RaiM feat. Artur & Adil - Дом бе...
2,D000000003,muzvuk.net/1965-raim-ft-artur-amp-adil-dom.html,RaiM ft. Artur & Adil - Дом - Скачать mp3,Скачать песню RaiM ft. Artur & Adil - Дом в Фо...


In [4]:
def load_queries(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", header=None,
                       names=["QueryId", "QueryText"], encoding="utf-8")

def load_qrels(path: Path) -> pd.DataFrame:
    return pd.read_csv(
        path, sep=r"\s+", header=None,
        names=["QueryId", "Q0", "DocumentId", "Label"],
        encoding="utf-8",
    )[["QueryId", "DocumentId", "Label"]]

train_queries = load_queries(TRAIN_Q_PATH)
train_qrels = load_qrels(TRAIN_QRELS_PATH)

print("train_queries:", train_queries.shape)
print("train_qrels:", train_qrels.shape)
train_queries.head(3), train_qrels.head(3)

train_queries: (1000, 2)
train_qrels: (25154, 3)


(   QueryId                                          QueryText
 0        2                                        0 16 0 8t t
 1        8  1 3 состав ассортимента холодных блюд и закусо...
 2       19  1 функциональное назначение продукции для каки...,
    QueryId  DocumentId  Label
 0        2  D000000015      1
 1        2  D000000016      2
 2        2  D000000017      1)

# **EDA**

In [5]:
print("Уникальных запросов в train:", train_queries.QueryId.nunique())
print("Уникальных документов в qrels:", train_qrels.DocumentId.nunique())
print("Уникальных документов в docs:", docs.DocumentId.nunique())

label_counts = train_qrels.Label.value_counts().sort_index()
print("\nРаспределение Label:")
print(label_counts)
print("\nДоли:")
print((label_counts / label_counts.sum()).round(3))

docs_per_query = train_qrels.groupby("QueryId").size()
print("\nДокументов на запрос (qrels):")
print(docs_per_query.describe())

Уникальных запросов в train: 1000
Уникальных документов в qrels: 25089
Уникальных документов в docs: 74848

Распределение Label:
Label
0     1205
1    12129
2     8170
3     3650
Name: count, dtype: int64

Доли:
Label
0    0.048
1    0.482
2    0.325
3    0.145
Name: count, dtype: float64

Документов на запрос (qrels):
count    1000.000000
mean       25.154000
std         6.782646
min        10.000000
25%        20.000000
50%        26.000000
75%        30.000000
max        49.000000
dtype: float64


In [6]:
print("пустых Title:", (docs.Title.fillna("").str.strip() == "").mean().round(4))
print("пустых Body :", (docs.Body.fillna("").str.strip() == "").mean().round(4))
print("max Body chars:", docs.Body.fillna("").str.len().max())

пустых Title: 0.0
пустых Body : 0.0
max Body chars: 126655


In [7]:
sample_qids = train_qrels.QueryId.drop_duplicates().sample(3, random_state=SEED).tolist()

for qid in sample_qids:
    qtext = train_queries.loc[train_queries.QueryId == qid, "QueryText"].iloc[0]
    print("=" * 100)
    print(f"QueryId={qid} | {qtext}")
    sub = train_qrels[train_qrels.QueryId == qid].merge(
        docs[["DocumentId", "Title"]], on="DocumentId", how="left"
    ).sort_values("Label", ascending=False)
    for _, row in sub.head(8).iterrows():
        title = (row.Title or "")[:120]
        print(f"  label={row.Label} | {title}")

QueryId=4031 | департамент дошкольного образования томска официальный сайт запись на консультацию
  label=2 | Основные сведения
  label=2 | Отдел методического сопровождения дошкольного, общего, специального и дополнительного образования — МАУ ИМЦ
  label=2 | департамент дошкольного образования - Результаты поиска - Cправочник организаций - Томск - tomsk.ru09.ru
  label=2 | Комитет по дошкольному образованию Администрации г. Томска
  label=2 | Комитет по дошкольному образованию Администрации г. Томска на ул. Розы Люксембург, 8 в Томске — телефон, схема проезда, 
  label=1 | Сайт Управления образования Чаинского района - Главная страница
  label=1 | Официальный портал муниципального образования 'Город Томск': Дошкольное образование
  label=1 | Срочно работа: Педагогическое образование в Томске — Февраль 2022 — 6419+ вакансий - Jooble
QueryId=5511 | как в домашних условиях убрать обвисший подбородок
  label=3 | Как убрать второй подбородок и щеки быстро без операции - упражнениями!
  lab

# **Эксперименты**

In [8]:
def ndcg_at_k(rels_sorted: np.ndarray, k: int = 10) -> float:
    rels = np.asarray(rels_sorted, dtype=float)[:k]
    if rels.size == 0:
        return 0.0
    gains = (2.0 ** rels - 1.0)
    discounts = np.log2(np.arange(2, rels.size + 2))
    dcg = (gains / discounts).sum()
    ideal = np.sort(rels)[::-1][:k]
    gains_i = (2.0 ** ideal - 1.0)
    disc_i = np.log2(np.arange(2, ideal.size + 2))
    idcg = (gains_i / disc_i).sum()
    return float(dcg / idcg) if idcg > 0 else 0.0


def evaluate_run(run_df: pd.DataFrame, qrels_df: pd.DataFrame, k: int = 10) -> float:
    merged = run_df.merge(qrels_df, on=["QueryId", "DocumentId"], how="left")
    merged["Label"] = merged["Label"].fillna(0).astype(int)
    merged = merged.sort_values(["QueryId", "Score", "DocumentId"],
                                ascending=[True, False, True])
    scores = []
    for qid, grp in merged.groupby("QueryId", sort=False):
        scores.append(ndcg_at_k(grp.Label.values, k=k))
    return float(np.mean(scores)) if scores else 0.0


RESULTS_COLUMNS = [
    "name",
    "config",
    "full_train_ndcg",
    "cv_mean",
    "cv_std",
]

results_df = pd.DataFrame(columns=RESULTS_COLUMNS)

## **TF-IDF бейзлайн**

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
import time

t0 = time.time()
docs["Text"] = docs.Title.fillna("") + " " + docs.Body.fillna("")

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
doc_matrix = vectorizer.fit_transform(docs.Text)
print("doc_matrix shape:", doc_matrix.shape)
print(f"vectorize elapsed: {time.time() - t0:.1f}s")

doc_id_to_idx = {d: i for i, d in enumerate(docs.DocumentId)}


def score_tfidf(queries_df, pairs_df, vectorizer, doc_matrix, doc_id_to_idx):
    q_texts = queries_df.set_index("QueryId").loc[pairs_df.QueryId.unique(), "QueryText"]
    q_matrix = vectorizer.transform(q_texts.values)
    results = []
    for qid, qvec in zip(q_texts.index, q_matrix):
        sub = pairs_df[pairs_df.QueryId == qid]
        doc_indices = [doc_id_to_idx[d] for d in sub.DocumentId if d in doc_id_to_idx]
        if not doc_indices:
            continue
        sims = cosine_similarity(qvec, doc_matrix[doc_indices]).ravel()
        for did, s in zip(sub.DocumentId.values, sims):
            results.append((qid, did, float(s)))
    return pd.DataFrame(results, columns=["QueryId", "DocumentId", "Score"])


t0 = time.time()
run_tfidf = score_tfidf(train_queries, train_qrels, vectorizer, doc_matrix, doc_id_to_idx)
ndcg_full = evaluate_run(run_tfidf, train_qrels, k=10)
print(f"full-train nDCG@10 = {ndcg_full:.4f}  ({time.time() - t0:.1f}s)")

unique_qids = train_qrels.QueryId.unique()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores = []
for fold_idx, (_, val_idx) in enumerate(kf.split(unique_qids)):
    val_qids = unique_qids[val_idx]
    run_val = score_tfidf(
        train_queries,
        train_qrels[train_qrels.QueryId.isin(val_qids)],
        vectorizer, doc_matrix, doc_id_to_idx,
    )
    fold_scores.append(evaluate_run(run_val, train_qrels, k=10))
    print(f"fold {fold_idx}: nDCG@10 = {fold_scores[-1]:.4f}")

fold_scores = np.array(fold_scores)

results_df.loc[len(results_df)] = {
    "name": "tfidf_baseline",
    "config": {
        "vectorizer": "TfidfVectorizer",
        "fields": "Title + ' ' + Body",
        "sublinear_tf": True,
        "min_df": 2,
        "max_df": 0.95,
        "token_pattern": r"(?u)\b\w+\b",
        "similarity": "cosine",
        "lemmatization": False,
    },
    "full_train_ndcg": ndcg_full,
    "cv_mean": fold_scores.mean(),
    "cv_std": fold_scores.std()
}

results_df

doc_matrix shape: (74848, 752221)
vectorize elapsed: 66.2s
full-train nDCG@10 = 0.8109  (2.9s)
fold 0: nDCG@10 = 0.8110
fold 1: nDCG@10 = 0.8117
fold 2: nDCG@10 = 0.8247
fold 3: nDCG@10 = 0.8070
fold 4: nDCG@10 = 0.8003


,name,config,full_train_ndcg,cv_mean,cv_std
0,tfidf_baseline,"{'vectorizer': 'TfidfVectorizer', 'fields': 'T...",0.810941,0.810941,0.008014


## **TF-IDF по леммам**

In [11]:
import pymorphy3
import pickle
import re
import time
from pathlib import Path

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
LEMMA_PATH = CACHE_DIR / "docs_lemmatized.pkl"

morph = pymorphy3.MorphAnalyzer()

TOKEN_RE = re.compile(r"[а-яёa-z0-9]+", re.IGNORECASE)

_lemma_cache = {}

def lemmatize_text(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    tokens = TOKEN_RE.findall(text.lower())
    lemmas = []
    for tok in tokens:
        lem = _lemma_cache.get(tok)
        if lem is None:
            lem = morph.parse(tok)[0].normal_form
            _lemma_cache[tok] = lem
        lemmas.append(lem)
    return " ".join(lemmas)


if LEMMA_PATH.exists():
    print("loading lemmatized docs from cache...")
    with open(LEMMA_PATH, "rb") as f:
        docs_lemma = pickle.load(f)
    print(f"loaded: {docs_lemma.shape}")
else:
    print("lemmatizing docs (may take several minutes)...")
    t0 = time.time()
    docs_lemma = docs[["DocumentId"]].copy()
    docs_lemma["Title_lemma"] = [lemmatize_text(t) for t in tqdm(docs.Title, desc="title")]
    print(f"  title done: {time.time()-t0:.1f}s")
    docs_lemma["Body_lemma"] = [lemmatize_text(b) for b in tqdm(docs.Body, desc="body")]
    print(f"  body done: {time.time()-t0:.1f}s")
    with open(LEMMA_PATH, "wb") as f:
        pickle.dump(docs_lemma, f)
    print(f"saved to {LEMMA_PATH}")

docs_lemma.head(3)

loading lemmatized docs from cache...
loaded: (74848, 3)


,DocumentId,Title_lemma,Body_lemma
0,D000000001,дом raim feat artur adil музыка в mp3 скачать ...,скачивать и слушать raim дом o2 2018 на zvuk t...
1,D000000002,скачать raim feat artur adil дом 2018,скачать песня raim feat artur adil дом бесплат...
2,D000000003,raim ft artur adil дом скачать mp3,скачать песня raim ft artur adil дом в формат ...


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

t0 = time.time()

docs_lemma["Text_lemma"] = docs_lemma.Title_lemma.fillna("") + " " + docs_lemma.Body_lemma.fillna("")

vectorizer_lemma = TfidfVectorizer(
    lowercase=False,          
    token_pattern=r"(?u)\b\w+\b",
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
doc_matrix_lemma = vectorizer_lemma.fit_transform(docs_lemma.Text_lemma)
print("doc_matrix_lemma shape:", doc_matrix_lemma.shape)
print(f"vectorize elapsed: {time.time()-t0:.1f}s")

doc_id_to_idx_lemma = {d: i for i, d in enumerate(docs_lemma.DocumentId)}


def score_tfidf_generic(queries_df, pairs_df, vectorizer, doc_matrix, doc_id_to_idx,
                        lemmatize=False):
    qids = pairs_df.QueryId.unique()
    q_texts = queries_df.set_index("QueryId").loc[qids, "QueryText"]
    if lemmatize:
        q_texts = q_texts.map(lemmatize_text)
    q_matrix = vectorizer.transform(q_texts.values)

    results = []
    for qid, qvec in zip(q_texts.index, q_matrix):
        sub = pairs_df[pairs_df.QueryId == qid]
        doc_indices = [doc_id_to_idx[d] for d in sub.DocumentId if d in doc_id_to_idx]
        if not doc_indices:
            continue
        sims = cosine_similarity(qvec, doc_matrix[doc_indices]).ravel()
        for did, s in zip(sub.DocumentId.values, sims):
            results.append((qid, did, float(s)))
    return pd.DataFrame(results, columns=["QueryId", "DocumentId", "Score"])


t0 = time.time()
run_lemma = score_tfidf_generic(
    train_queries, train_qrels, vectorizer_lemma, doc_matrix_lemma,
    doc_id_to_idx_lemma, lemmatize=True,
)
ndcg_full_lemma = evaluate_run(run_lemma, train_qrels, k=10)
print(f"\nfull-train nDCG@10 = {ndcg_full_lemma:.4f}  ({time.time()-t0:.1f}s)")

unique_qids = train_qrels.QueryId.unique()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores_lemma = []
for fold_idx, (_, val_idx) in enumerate(kf.split(unique_qids)):
    val_qids = unique_qids[val_idx]
    run_val = score_tfidf_generic(
        train_queries,
        train_qrels[train_qrels.QueryId.isin(val_qids)],
        vectorizer_lemma, doc_matrix_lemma, doc_id_to_idx_lemma,
        lemmatize=True,
    )
    fold_scores_lemma.append(evaluate_run(run_val, train_qrels, k=10))
    print(f"fold {fold_idx}: nDCG@10 = {fold_scores_lemma[-1]:.4f}")

fold_scores_lemma = np.array(fold_scores_lemma)
print(f"\nCV nDCG@10: mean = {fold_scores_lemma.mean():.4f}, std = {fold_scores_lemma.std():.4f}")

results_df.loc[len(results_df)] = {
    "name": "tfidf_lemma",
    "config": {
        "vectorizer": "TfidfVectorizer",
        "fields": "lemmatized Title + ' ' + lemmatized Body",
        "sublinear_tf": True,
        "min_df": 2,
        "max_df": 0.95,
        "token_pattern": r"(?u)\b\w+\b",
        "similarity": "cosine",
        "lemmatization": True,
        "lemmatizer": "pymorphy3",
    },
    "full_train_ndcg": ndcg_full_lemma,
    "cv_mean": fold_scores_lemma.mean(),
    "cv_std": fold_scores_lemma.std(),
}

results_df

doc_matrix_lemma shape: (74848, 415511)
vectorize elapsed: 46.8s

full-train nDCG@10 = 0.8126  (2.4s)
fold 0: nDCG@10 = 0.8126
fold 1: nDCG@10 = 0.8114
fold 2: nDCG@10 = 0.8236
fold 3: nDCG@10 = 0.8054
fold 4: nDCG@10 = 0.8100

CV nDCG@10: mean = 0.8126, std = 0.0060


,name,config,full_train_ndcg,cv_mean,cv_std
0,tfidf_baseline,"{'vectorizer': 'TfidfVectorizer', 'fields': 'T...",0.810941,0.810941,0.008014
1,tfidf_lemma,"{'vectorizer': 'TfidfVectorizer', 'fields': 'l...",0.812594,0.812594,0.006005


In [ ]:
!free -h

## **BM25F**

In [14]:
from collections import Counter
import numpy as np
import time
import pickle

t0 = time.time()

def tokenize(s):
    return s.split() if isinstance(s, str) else []

n_docs = len(docs_lemma)

title_lens = np.array([len(tokenize(t)) for t in docs_lemma.Title_lemma], dtype=np.float32)
body_lens  = np.array([len(tokenize(b)) for b in docs_lemma.Body_lemma],  dtype=np.float32)
avg_title_len = float(title_lens.mean())
avg_body_len  = float(body_lens.mean())
print(f"avg_title_len={avg_title_len:.2f}, avg_body_len={avg_body_len:.2f}")

title_tf = [Counter(tokenize(t)) for t in tqdm(docs_lemma.Title_lemma, desc="title_tf")]
body_tf  = [Counter(tokenize(b)) for b in tqdm(docs_lemma.Body_lemma,  desc="body_tf")]

title_df = Counter()
body_df  = Counter()
for c in title_tf:
    title_df.update(c.keys())
for c in body_tf:
    body_df.update(c.keys())

def idf(df, n):
    return np.log((n - df + 0.5) / (df + 0.5) + 1.0)

all_terms = set(title_df) | set(body_df)
idf_cache = {t: idf(title_df.get(t, 0) + body_df.get(t, 0), n_docs) for t in all_terms}

print(f"уникальных термов: {len(all_terms)}")
print(f"stats elapsed: {time.time()-t0:.1f}s")

with open(CACHE_DIR / "bm25f_stats.pkl", "wb") as f:
    pickle.dump({
        "title_tf": title_tf,
        "body_tf": body_tf,
        "title_lens": title_lens,
        "body_lens": body_lens,
        "avg_title_len": avg_title_len,
        "avg_body_len": avg_body_len,
        "idf_cache": idf_cache,
    }, f)
print("bm25f stats saved")

gc.collect()

avg_title_len=10.10, avg_body_len=1071.47


body_tf: 100%|██████████| 74848/74848 [00:30<00:00, 2479.79it/s]


уникальных термов: 1136127
stats elapsed: 49.3s


: 

In [ ]:
def bm25f_score_query(q_terms, doc_indices, title_tf, body_tf, title_lens, body_lens,
                      avg_title_len, avg_body_len, idf_cache,
                      k1=1.5, b_title=0.5, b_body=0.75, w_title=3.0, w_body=1.0):
    scores = np.zeros(len(doc_indices), dtype=np.float32)
    for pos, di in enumerate(doc_indices):
        t_lens = title_lens[di]
        b_lens = body_lens[di]
        t_tf = title_tf[di]
        b_tf = body_tf[di]
        s = 0.0
        for term in q_terms:
            idf_t = idf_cache.get(term, 0.0)
            if idf_t <= 0:
                continue
            tf_t = t_tf.get(term, 0)
            tf_b = b_tf.get(term, 0)
            tf_tilde = (
                w_title * tf_t / (1 - b_title + b_title * t_lens / avg_title_len)
                + w_body  * tf_b / (1 - b_body  + b_body  * b_lens / avg_body_len)
            )
            if tf_tilde > 0:
                s += idf_t * tf_tilde * (k1 + 1) / (tf_tilde + k1)
        scores[pos] = s
    return scores


def score_bm25f(queries_df, pairs_df, docs_lemma,
                title_tf, body_tf, title_lens, body_lens,
                avg_title_len, avg_body_len, idf_cache,
                k1=1.5, b_title=0.5, b_body=0.75, w_title=3.0, w_body=1.0,
                lemmatize=True):
    qids = pairs_df.QueryId.unique()
    q_texts = queries_df.set_index("QueryId").loc[qids, "QueryText"]
    if lemmatize:
        q_texts = q_texts.map(lemmatize_text)

    doc_id_to_idx_lemma = {d: i for i, d in enumerate(docs_lemma.DocumentId)}
    pairs_by_qid = {qid: sub for qid, sub in pairs_df.groupby("QueryId")}

    results = []
    for qid, qtext in tqdm(zip(q_texts.index, q_texts.values), total=len(q_texts), desc="bm25f"):
        sub = pairs_by_qid.get(qid)
        if sub is None:
            continue
        q_terms = qtext.split()
        doc_indices = [doc_id_to_idx_lemma[d] for d in sub.DocumentId if d in doc_id_to_idx_lemma]
        if not doc_indices:
            continue
        scores = bm25f_score_query(
            q_terms, doc_indices, title_tf, body_tf, title_lens, body_lens,
            avg_title_len, avg_body_len, idf_cache,
            k1=k1, b_title=b_title, b_body=b_body, w_title=w_title, w_body=w_body,
        )
        for did, s in zip(sub.DocumentId.values, scores):
            results.append((qid, did, float(s)))
    return pd.DataFrame(results, columns=["QueryId", "DocumentId", "Score"])

In [ ]:
from sklearn.model_selection import KFold

params = dict(k1=1.5, b_title=0.5, b_body=0.75, w_title=3.0, w_body=1.0)

t0 = time.time()
run_bm25f = score_bm25f(
    train_queries, train_qrels, docs_lemma,
    title_tf, body_tf, title_lens, body_lens,
    avg_title_len, avg_body_len, idf_cache, **params,
)
ndcg_full_bm25f = evaluate_run(run_bm25f, train_qrels, k=10)
print(f"full-train nDCG@10 = {ndcg_full_bm25f:.4f}  ({time.time()-t0:.1f}s)")

unique_qids = train_qrels.QueryId.unique()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores_bm25f = []
for fold_idx, (_, val_idx) in enumerate(kf.split(unique_qids)):
    val_qids = unique_qids[val_idx]
    run_val = score_bm25f(
        train_queries, train_qrels[train_qrels.QueryId.isin(val_qids)], docs_lemma,
        title_tf, body_tf, title_lens, body_lens,
        avg_title_len, avg_body_len, idf_cache, **params,
    )
    fold_scores_bm25f.append(evaluate_run(run_val, train_qrels, k=10))
    print(f"fold {fold_idx}: nDCG@10 = {fold_scores_bm25f[-1]:.4f}")

fold_scores_bm25f = np.array(fold_scores_bm25f)
print(f"\nCV nDCG@10: mean = {fold_scores_bm25f.mean():.4f}, std = {fold_scores_bm25f.std():.4f}")

results_df.loc[len(results_df)] = {
    "name": "bm25f_base",
    "config": {**params, "lemmatization": True, "idf": "smooth, combined df"},
    "full_train_ndcg": ndcg_full_bm25f,
    "cv_mean": fold_scores_bm25f.mean(),
    "cv_std": fold_scores_bm25f.std(),
}
results_df

# **Резко перестало хватать памяти джупитерноутбука поэтому решено переписать все через скрипты**

**По итогу имеем:**

| Модель | full_train nDCG@10 | CV mean | CV std | Δ к baseline |
|--------|-------------------|---------|--------|--------------|
| tfidf_baseline | 0.8109 | 0.8109 | 0.0080 | — |
| tfidf_lemma | 0.8126 | 0.8126 | 0.0060 | +0.0017 |
| bm25f_base | 0.8219 | 0.8219 | 0.0053 | +0.0110 |
| bm25f_tuned | 0.8271 | 0.8271 | 0.0070 | +0.0162 |
| **bm25f_tuned2** | **0.8283** | **0.8283** | **0.0068** | **+0.0174** |
| lm_dirichlet | 0.8082 | 0.8082 | 0.0076 | −0.0027 |
| lm_dirichlet2 | 0.8208 | 0.8208 | 0.0044 | +0.0099 |
| fusion_bm25f_lm | 0.8246 | 0.8246 | 0.0071 | +0.0137 |